In [6]:
# Imports

import numpy as np
import pandas as pd
import os
import glob
from datetime import datetime
from scipy.stats import skew

In [7]:
# Tuning Room

USE_DUMMY_DATA = False
EXPORT_TO_TXT = True
EXPORT_ALL_FEATURES_TO_CSV = False
COPY_MATCHING_SECTION_FILES = True

SIMILARITY_THRESHOLD = 60.0             # minimum certainty percentage to be included in the report
STRAIGHT_RADIUS_THRESHOLD = 500.0       # large radius to be consideres a straight and not corner
MAX_MATCHES_PER_SECTION = 3
USE_LOG_SCALING_FOR_ALL_FEATURES = True
FEATURE_WEIGHTS = {
    # entry/exit & length
    "entry_straight_length": 0.7,       # Prioritises a similar corner entry length
    "exit_straight_length": 0.7,        # Prioritises a similar corner exit length
    "cornering_percentage": 0.8,        # Prioritises a similar ratio of cornering vs straight time
    # corner shape
    "mean_corner_radius": 1.0,          # Prioritises a similar average corner speed
    "min_corner_radius": 1.0,           # Prioritises a similar apex sharpness / minimum speed
    "std_dev_corner_radius": 0.8,       # Prioritises a similar arc consistency, if variable or constant
    "radius_skewness": 0.8,             # Prioritises a similar shape asymmetry between entry and exit
    "direction_changes": 0.0,           # Prioritises a similar number of direction changes
    # elevation
    "net_elevation_change": 0.2,        # Prioritises a similar uphill or downhill slope
    "total_climb_and_descent": 0.2,     # Prioritises a similar amount of undulation
}

In [8]:
# Helper Functions

# defines output file name
def get_next_report_filename(report_dir, target_name):
    os.makedirs(report_dir, exist_ok=True)
    base_name = f"analysis_{target_name}_"
    file_pattern = os.path.join(report_dir, f"{base_name}*.txt")
    existing_files = glob.glob(file_pattern)
    
    if not existing_files:
        return os.path.join(report_dir, f"{base_name}01.txt")  
    
    max_num = 0
    for f in existing_files:
        try:
            num_str = os.path.basename(f).replace(base_name, '').replace('.txt', '')
            num = int(num_str)
            if num > max_num:
                max_num = num
        except ValueError:
            continue
    
    next_num = max_num + 1
    return os.path.join(report_dir, f"{base_name}{next_num:02d}.txt")

# calculates the corner radius
def calculate_signed_radius(p1, p2, p3):
    p1, p2, p3 = np.array(p1), np.array(p2), np.array(p3)

    numerator = 2 * ((p1[0] - p3[0]) * (p2[1] - p1[1]) - (p1[0] - p2[0]) * (p3[1] - p1[1]))
    denominator = np.linalg.norm(p1 - p2) * np.linalg.norm(p2 - p3) * np.linalg.norm(p3 - p1)

    if abs(denominator) < 1e-9:  # collinear points
        return float('inf')
    
    curvature = numerator / denominator

    if abs(curvature) < 1e-9:    # nearly zero curvature
        return float('inf')
    
    return 1/curvature

# # --- START OF MISSING CODE BLOCK 1 ---
# # This is the debug function that prints the validation table for 'direction_changes'.
# def print_debug_table(target_name, target_data, training_data):
#     """Prints a table of direction changes for all track sections to validate logic."""
#     print("\n--- [DEBUG] Direction Change Validation Table ---")
#     print(f"{'Track':<20} | {'Section ID':<12} | {'Direction Changes'}")
#     print("-" * 55)
#     # Print target track data
#     for sec_id, data in sorted(target_data.items()):
#         changes = data['features']['direction_changes']
#         print(f"{target_name:<20} | {sec_id:<12} | {changes}")
#     # Print training track data
#     for track_name, sections in training_data.items():
#         for sec_id, data in sorted(sections.items()):
#             changes = data['features']['direction_changes']
#             print(f"{track_name:<20} | {sec_id:<12} | {changes}")
#     print("-" * 55)
# # --- END OF MISSING CODE BLOCK 1 ---

In [9]:
# Execution

# calculation of a descriptive feature vector for a track section
class FeatureExtractor:
    # UPDATED: The class now requires the manual values to be provided.
    def __init__(self, full_track_coords, section_start_m, section_end_m, 
                 manual_dir_changes, manual_entry_straight, manual_exit_straight):
        self.full_track_coords = full_track_coords
        self.section_start_m = section_start_m
        self.section_end_m = section_end_m
        # Store the manually provided values.
        self.manual_dir_changes = manual_dir_changes
        self.manual_entry_straight = manual_entry_straight
        self.manual_exit_straight = manual_exit_straight

        diffs = np.diff(self.full_track_coords, axis=0)
        seg_lengths = np.linalg.norm(diffs, axis=1)
        self.dist_along_track = np.insert(np.cumsum(seg_lengths), 0, 0)
        
        self.section_coords = self._get_section_coordinates()

        if len(self.section_coords) < 3:
            self.features = None
            return
        self.distances, self.radii, self.elevation_changes = self._calculate_point_properties()
        self.features = self._extract_all_features()
    
    # slices coordinates into sections (This method is unchanged)
    def _get_section_coordinates(self):
        start_m, end_m = self.section_start_m, self.section_end_m
        if start_m <= end_m:
            start_idx = np.searchsorted(self.dist_along_track, start_m, side='left')
            end_idx = np.searchsorted(self.dist_along_track, end_m, side='right')
            return self.full_track_coords[start_idx:end_idx]
        else:
            start_idx = np.searchsorted(self.dist_along_track, start_m, side='left')
            part1 = self.full_track_coords[start_idx:]
            end_idx = np.searchsorted(self.dist_along_track, end_m, side='right')
            part2 = self.full_track_coords[:end_idx]
            return np.concatenate((part1, part2), axis=0)
    
    # calculates radius and elevation changes for each point in a section (This method is unchanged)
    def _calculate_point_properties(self):
        coords_2d = self.section_coords[:, :2]
        distances = [np.linalg.norm(self.section_coords[i+1] - self.section_coords[i]) for i in range(len(self.section_coords)-1)]
        radii = [float('inf')]
        for i in range(1, len(coords_2d) - 1):
            radii.append(calculate_signed_radius(coords_2d[i-1], coords_2d[i], coords_2d[i+1]))
        radii.append(float('inf'))
        elevation_changes = np.diff(self.section_coords[:, 2])
        return distances, np.array(radii), elevation_changes
    
    # computes the final feature vector (Now much simpler)
    def _extract_all_features(self):
        # Directly use the manual values provided.
        entry_straight_len = self.manual_entry_straight
        exit_straight_len = self.manual_exit_straight
        dir_changes = self.manual_dir_changes

        # The rest of the features are still calculated from the geometry.
        total_section_length = sum(self.distances)
        if total_section_length == 0: total_section_length = 1e-6
        
        is_cornering = np.abs(self.radii) < STRAIGHT_RADIUS_THRESHOLD
        corner_indices = np.where(is_cornering)[0]
        cornering_dist = sum(self.distances[i] for i in corner_indices if i< len(self.distances))
        cornering_perc = (cornering_dist / total_section_length) * 100.0

        corner_radii = self.radii[is_cornering]
        corner_radii = corner_radii[np.isfinite(corner_radii)]
        
        mean_rad, min_rad, std_rad, skew_rad = (0, 0, 0, 0)
        
        if len(corner_radii) > 2:
            abs_corner_radii = np.abs(corner_radii)
            mean_rad, min_rad, std_rad = np.mean(abs_corner_radii), np.min(abs_corner_radii), np.std(abs_corner_radii)
            skew_rad = skew(abs_corner_radii)

        net_elev_change = self.section_coords[-1, 2] - self.section_coords[0, 2]
        total_climb_descent = np.sum(np.abs(self.elevation_changes))

        # This method now ONLY returns the raw, physical values. No scaling happens here.
        return{
            "entry_straight_length": entry_straight_len, "exit_straight_length": exit_straight_len,
            "cornering_percentage": cornering_perc, "mean_corner_radius": mean_rad,
            "min_corner_radius": min_rad, "std_dev_corner_radius": std_rad, "radius_skewness": skew_rad,
            "direction_changes": dir_changes, "net_elevation_change": net_elev_change, "total_climb_and_descent": total_climb_descent 
        }

In [10]:
#--- EXECUTION ---

def main():
    print("Starting Track Analysis Script...")

    # detect target track and load data
    data_type = "Dummy" if USE_DUMMY_DATA else "Real"
    base_path = os.getcwd()
    coords_path = os.path.join(base_path, "Coordinates")
    sections_path = os.path.join(base_path, "Sections", data_type)

    print(f"Scanning for target track in '{sections_path}'...")
    all_section_files = glob.glob(os.path.join(sections_path, "*.csv"))
    target_section_files = [f for f in all_section_files if "_target_" in os.path.basename(f)]

    if len(target_section_files) != 1:
        print(f"ERROR: Found {len(target_section_files)} target files. Expected 1.")
        print("A target file must contain '_target' in its name( e.g., 'sections_target_zandvoort.csv').")
        return
    
    target_section_file_path = target_section_files[0]
    target_section_filename = os.path.basename(target_section_file_path)
    TARGET_TRACK_NAME = target_section_filename.replace('sections_target_', '').replace('.csv', '')
    print(f"Detected Target Track: '{TARGET_TRACK_NAME}'")

    try:
        target_coords_df = pd.read_csv(
            os.path.join(coords_path, f"{TARGET_TRACK_NAME}.csv"), 
            header=None, 
            dtype=np.float64, 
            skiprows=1 
        )
        target_sections_df = pd.read_csv(target_section_file_path)
    except FileNotFoundError as e:
        print(f"\nERROR: Could not find all necessary files for '{TARGET_TRACK_NAME}'. Missing: {e.filename}")
        return
    
    training_coords, training_sections = {}, {}
    for f in all_section_files:
        if f == target_section_file_path: continue
        track_name = os.path.basename(f).replace('sections_', '').replace('.csv', '')
        try:
            training_coords[track_name] = pd.read_csv(
                os.path.join(coords_path, f"{track_name}.csv"), 
                header=None, 
                dtype=np.float64, 
                skiprows=1
            )
            training_sections[track_name] = pd.read_csv(f)
        except FileNotFoundError:
            print(f"Warning: Skipping '{track_name}' due to missing coordinates or sections file.")

    if not training_sections:
        print("ERROR: No training tracks were found. Exiting.")
        return

    print(f"Loaded Target: '{TARGET_TRACK_NAME}' and {len(training_sections)} training tracks.")
    
    # feature extraction
    print("Extracting features from all track sections...")
    all_features = []
    target_track_data={}
    numeric_features_for_analysis = []
    target_coords_np=target_coords_df.to_numpy()

    for i,row in target_sections_df.iterrows():
        manual_changes = int(row['dir_changes'])
        manual_entry = float(row['entry_straight'])
        manual_exit = float(row['exit_straight'])

        extractor=FeatureExtractor(target_coords_np, row['start_m'], row['end_m'],
                                   manual_dir_changes=manual_changes,
                                   manual_entry_straight=manual_entry,
                                   manual_exit_straight=manual_exit)
        if extractor.features:
            numeric_features_for_analysis.append(extractor.features)
            feature_dict_for_export = extractor.features.copy()
            feature_dict_for_export['track_name'] = TARGET_TRACK_NAME
            feature_dict_for_export['segment_id'] = int(row['segment_id'])
            all_features.append(feature_dict_for_export)
            target_track_data[int(row['segment_id'])] = {"features": extractor.features,"start_m": row['start_m'], "end_m": row['end_m']}
        else:
            print(f"  -> WARNING: Skipping Target Section {int(row['segment_id'])} (track data likely too short for analysis).")
            
    # UPDATED: Training track loop now reads the new columns and no longer needs the fallback logic.
    training_tracks_data = {}
    for name, sections_df in training_sections.items():
        training_tracks_data[name] = {}
        coords_np = training_coords[name].to_numpy()
        for i,row in sections_df.iterrows():
            manual_changes = int(row['dir_changes'])
            manual_entry = float(row['entry_straight'])
            manual_exit = float(row['exit_straight'])
            
            extractor=FeatureExtractor(coords_np, row['start_m'], row['end_m'], 
                                       manual_dir_changes=manual_changes, 
                                       manual_entry_straight=manual_entry, 
                                       manual_exit_straight=manual_exit)
            
            if extractor.features:
                numeric_features_for_analysis.append(extractor.features)
                feature_dict_for_export = extractor.features.copy()
                feature_dict_for_export['track_name'] = name
                feature_dict_for_export['segment_id'] = int(row['segment_id'])
                all_features.append(feature_dict_for_export)
                training_tracks_data[name][int(row['segment_id'])] = {
                    "features": extractor.features,
                    "start_m": row['start_m'],
                    "end_m": row['end_m']
                }
            else:
                print(f"  -> WARNING: Skipping Section {int(row['segment_id'])} of {name} (track data likely too short for analysis).")

    # This line calls the debug function to print the validation table.
    # print_debug_table(TARGET_TRACK_NAME, target_track_data, training_tracks_data)
    # --- END OF MISSING CODE BLOCK 2 ---
    
    # Export feature tables
    if EXPORT_ALL_FEATURES_TO_CSV and all_features:
        export_df = pd.DataFrame(all_features)

        cols = ['track_name', 'segment_id'] + [c for c in export_df.columns if c not in ['track_name', 'segment_id']]
        export_df = export_df[cols]

        output_filename = f"all_features_export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

        report_dir = os.path.join(sections_path, "Reports")
        os.makedirs(report_dir, exist_ok=True)
        output_path = os.path.join(report_dir, output_filename)

        export_df.to_csv(output_path, index=False)
        print(f"\nSuccessfully exported all features to '{output_path}'.")

    # feature normalisation
    features_df = pd.DataFrame(numeric_features_for_analysis)
    
    # Step 2: Create a completely separate DataFrame for the mathematical analysis.
    analysis_df = features_df.copy()

    # Step 3: If the global flag is set, apply log scaling to all appropriate columns
    # IN THE ANALYSIS DATAFRAME ONLY.
    if USE_LOG_SCALING_FOR_ALL_FEATURES:
        print("Applying logarithmic scaling to all physical features for analysis...")
        # Define which columns are safe to transform (must be non-negative).
        # We exclude percentages, counts, and features that can be negative.
        cols_to_transform = [
            'entry_straight_length', 'exit_straight_length', 
            'mean_corner_radius', 'min_corner_radius', 'std_dev_corner_radius',
            'total_climb_and_descent'
        ]
        for col in cols_to_transform:
            analysis_df[col] = np.log1p(analysis_df[col])

    # Step 4: Perform all normalization calculations on the transformed analysis_df.
    features_df_range = analysis_df.max() - analysis_df.min()
    features_df_range[features_df_range == 0] = 1
    normalised_features_df = ((analysis_df - analysis_df.min()) / features_df_range).fillna(0)

    # comparison and report generation
    print("Comparing sections and generating report...")
    
    report_lines = []
    unmatched_target_sections = []
    
    # ... (the report header generation is unchanged) ...
    report_lines.append("="*50 + "\n")

    norm_weights = np.array([FEATURE_WEIGHTS[col] for col in features_df.columns])
    norm_weights /= np.sum(norm_weights)
    max_dist = np.sqrt(np.sum(norm_weights))

    # The main comparison loop starts here. It uses simple counters to stay in sync.
    feature_counter = 0
    for target_sec_id, target_data in target_track_data.items():
        report_lines.append(f"--- Target Section: {target_sec_id} ({target_data['start_m']:.2f}m to {target_data['end_m']:.2f}m) ---")

        # This contains the ORIGINAL, RAW values for displaying in the report.
        target_features = pd.Series(target_data["features"])
        
        # This gets the corresponding row of TRANSFORMED & NORMALIZED data for the math.
        norm_target_features = normalised_features_df.iloc[feature_counter]
        feature_counter += 1

        matches = []
        train_feature_counter = len(target_track_data)
        for train_track, sections in training_tracks_data.items():
            for train_sec_id, train_features_dict in sections.items():
                
                # The "hard filter" correctly uses the ORIGINAL feature data for its check.
                if target_features['direction_changes'] != train_features_dict['features']['direction_changes']:
                    train_feature_counter += 1
                    continue
                
                norm_train_features = normalised_features_df.iloc[train_feature_counter]
                train_feature_counter += 1

                # similarity calculation
                diff = norm_target_features - norm_train_features
                weighted_dist_sq = np.sum(norm_weights * (diff **2))
                weighted_dist = np.sqrt(weighted_dist_sq)
                certainty = (1 - (weighted_dist / max_dist)) * 100.0

                if certainty >= SIMILARITY_THRESHOLD:
                    matches.append({
                        "train_track": train_track,
                        "train_sec_id": train_sec_id,
                        "certainty": certainty,
                        "raw_features": train_features_dict["features"],
                        "norm_features": norm_train_features
                    })
        
        matches.sort(key=lambda x: x['certainty'], reverse=True)

        matches = matches[:MAX_MATCHES_PER_SECTION]

        # build report
        if not matches:
            report_lines.append(f"Found 0 potential matches above the {SIMILARITY_THRESHOLD}% threshold.\n")
            unmatched_target_sections.append(target_sec_id)
        else:
            report_lines.append(f"Found {len(matches)} potential match(es):\n")
            # Loop through each valid match and create the detailed report table.
            for i, match in enumerate(matches):
                report_lines.append(f"[MATCH {i+1}]")
                report_lines.append(f"- Training Track:    {match['train_track']}")
                report_lines.append(f"- Section:           {match['train_sec_id']}")
                report_lines.append(f"- Overall Certainty: {match['certainty']:.1f}%\n")

                # Define the header for the feature breakdown table.
                table_header = "| {:<25} | {:<6} | {:<12} | {:<12} | {:<10} |".format("Feature Name", "Weight", "Target Val", "Train Val", "Similarity")
                table_line = "+{:-<27}+{:-<8}+{:-<14}+{:-<14}+{:-<12}+".format("", "", "", "", "")
                report_lines.append(table_line)
                report_lines.append(table_header)
                report_lines.append(table_line)

                feature_similarities = []
                for name, weight in FEATURE_WEIGHTS.items():
                    target_val = target_features[name]
                    train_val = match['raw_features'][name]

                    norm_target = norm_target_features[name]
                    norm_train = match['norm_features'][name]
                    
                    feat_sim = (1-abs(norm_target - norm_train)) * 100.0
                    feature_similarities.append({'name': name, 'sim': feat_sim})

                    if "radius" in name or "length" in name:
                        val_str_t, val_str_tr = f"{target_val:.1f} m", f"{train_val:.1f} m"
                    elif "percentage" in name:
                        val_str_t, val_str_tr = f"{target_val:.1f}%", f"{train_val:.1f}%"
                    else:
                        val_str_t, val_str_tr = f"{target_val:.2f}", f"{train_val:.2f}"
                    
                    row_str = "| {:<25} | {:<6.2f} | {:<12} | {:<12} | {:>9.1f}% |".format(name, weight, val_str_t, val_str_tr, feat_sim)
                    report_lines.append(row_str)
                
                report_lines.append(table_line)

                # Sort features by similarity to find the best and worst matching characteristics.
                feature_similarities.sort(key=lambda x: x['sim'])
                least_sim = feature_similarities[0]
                most_sim = feature_similarities[-1]
                report_lines.append(f"\n- Most Similar Feature:    {most_sim['name']} ({most_sim['sim']:.1f}% similar)")
                report_lines.append(f"- Most Dissimilar Feature: {least_sim['name']} ({least_sim['sim']:.1f}% similar)\n")

            # Copy sections to new folders
            if COPY_MATCHING_SECTION_FILES:

                match_data_for_csv = []
                for match in matches:
                    train_section_info = training_tracks_data[match['train_track']][match['train_sec_id']]
                    match_data_for_csv.append({
                        "segment_id": match['train_sec_id'],
                        "start_m": train_section_info['start_m'],
                        "end_m": train_section_info['end_m'],
                        "track_name": match['train_track'],
                        "certainty": match['certainty'] / 100.0
                    })
            
                matches_df = pd.DataFrame(match_data_for_csv)

                dest_folder_name = f"Target_{TARGET_TRACK_NAME.capitalize()}"
                destination_dir = os.path.join(sections_path, dest_folder_name)
                os.makedirs(destination_dir, exist_ok=True)
                    
                destination_filename = f"section{target_sec_id:02d}.csv"
                destination_file_path = os.path.join(destination_dir, destination_filename)

                should_save = True

                if os.path.exists(destination_file_path):
                    overwrite = input(f"WARNING: '{destination_file_path}' already exists. Overwrite? (y/n): ").lower().strip()
                    if overwrite in ['y', 'yes']:
                        print(f"   -> Overwriting '{os.path.basename(destination_file_path)}'...")
                        should_save = True
                    else:
                        print(f"   -> Skipping file '{os.path.basename(destination_file_path)}'.")
                        should_save = False
              
                if should_save:
                    if not os.path.exists(destination_file_path):
                        print(f"   -> Creating match file: '{destination_filename}'...")
                    matches_df.to_csv(destination_file_path, index=False)
        
        report_lines.append("="*50 + "\n")
    
    final_report = "\n".join(report_lines)
    print(final_report)

    # export
    if EXPORT_TO_TXT:
        report_dir = os.path.join(sections_path, "Reports")
        filename = get_next_report_filename(report_dir, TARGET_TRACK_NAME)

        with open(filename, 'w') as f:
            f.write(final_report)
        print(f"\nReport successfully exported to '{filename}'.")

    # report unmatched sections
    if unmatched_target_sections:
        print("\n" + "="*50)
        print("!!! ANALYSIS WARNING !!!")
        print("The following sections in the target track had NO matches above the threshold:")
        unmatched_str = ', '.join(map(str, sorted(unmatched_target_sections)))
        print(f"-> Sections: {unmatched_str}")
        print('='*50)


if __name__ == '__main__':
    # This block ensures that the required Python libraries are installed before running the main script.
    try:
        import numpy, pandas, scipy
    except ImportError as e:
        print(f"Error: Missing required library -> {e.name}. Please run: pip install numpy pandas scipy")
    else:
        main()

Starting Track Analysis Script...
Scanning for target track in 'h:\My Drive\Cranfield\11. Individual Thesis\Code\Sections\Real'...
Detected Target Track: 'monza'
Loaded Target: 'monza' and 6 training tracks.
Extracting features from all track sections...
Applying logarithmic scaling to all physical features for analysis...
Comparing sections and generating report...
   -> Overwriting 'section01.csv'...
   -> Overwriting 'section02.csv'...
   -> Overwriting 'section03.csv'...
   -> Overwriting 'section04.csv'...
   -> Overwriting 'section05.csv'...
   -> Overwriting 'section06.csv'...
   -> Overwriting 'section07.csv'...

--- Target Section: 1 (0.00m to 1181.46m) ---
Found 3 potential match(es):

[MATCH 1]
- Training Track:    donington_gp
- Section:           5
- Overall Certainty: 72.7%

+---------------------------+--------+--------------+--------------+------------+
| Feature Name              | Weight | Target Val   | Train Val    | Similarity |
+---------------------------+-------